# Unified Multi-Engine Reasoning Framework Demo

This notebook demonstrates the complete framework capability set built across this design branch:

1. **Schema** — Entity/Relationship/Field declarations
2. **Annotation Store** — Four-layer data architecture (Claim / Annotation / Legacy / Provenance)
3. **Multi-Engine Evaluate** — `Store.evaluate(mode=...)` with engine_ext + engine_options
4. **Provenance** — Engine-native carriers + ProvenanceEnvelope
5. **EvidenceGraph** — Unified cross-engine explainability representation
6. **Audit Package** — assertion_annotations.jsonl + evidence_graphs.jsonl

Each section runs **real framework code** — no mocks, no simulated output.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

# Ensure src/ is on the Python path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

from factpy_kernel.sdk import Entity, Identity, Field, Relationship
from factpy_kernel.sdk.store import SDKStore
from factpy_kernel.sdk.compile import compile_schema_from_classes
from factpy_kernel.sdk.dsl import Derivation, Pred, vars as sdk_vars
from factpy_kernel.core.evidence.write_protocol import set_field
from factpy_kernel.core.derivation.accept import AcceptOptions
from factpy_kernel.core.store.ledger import Ledger

print("Framework imports OK")

Framework imports OK


## 1. Schema: Entity + Relationship

All engines share the same schema. The framework compiles Entity/Relationship
classes into a `schema_ir` that each engine adapter can consume.

In [2]:
class Researcher(Entity):
    researcher_id: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")
    expertise: str = Field(cardinality="single")
    impact_score: str = Field(cardinality="single")


class Collaboration(Relationship):
    from_entity = Researcher
    to_entity = Researcher
    joint_papers: str = Field(cardinality="single")


schema_ir = compile_schema_from_classes([Researcher, Collaboration])
sdk = SDKStore([Researcher], schema_ir=schema_ir)

print(f"Schema: {len(schema_ir['predicates'])} predicates")
for pred in schema_ir['predicates']:
    rel = f" (relationship: {pred['relationship_type']})" if pred.get('relationship_type') else ""
    print(f"  {pred['pred_id']}{rel}")

Schema: 6 predicates
  Researcher:exists
  researcher:researcher_id
  researcher:name
  researcher:expertise
  researcher:impact_score
  collaboration:joint_papers (relationship: Collaboration)


## 2. Write Facts + Annotation Store

The write protocol dual-writes:
- Whitelisted meta → `annotation_rows` (canonical) + `meta_rows` (legacy)
- Custom meta → `meta_rows` only

This demonstrates the four-layer architecture in action.

In [3]:
# Write EDB facts
alice_ref = sdk.ref(Researcher, researcher_id="Alice")
bob_ref = sdk.ref(Researcher, researcher_id="Bob")
carol_ref = sdk.ref(Researcher, researcher_id="Carol")

for ref, name, expertise in [
    (alice_ref, "Alice Chen", "NLP"),
    (bob_ref, "Bob Zhang", "CV"),
    (carol_ref, "Carol Li", "RL"),
]:
    set_field(sdk.ledger, pred_id="researcher:name", e_ref=ref,
              rest_terms=[("string", name)],
              meta={"source": "faculty_db", "confidence": 1.0})
    set_field(sdk.ledger, pred_id="researcher:expertise", e_ref=ref,
              rest_terms=[("string", expertise)],
              meta={"source": "publication_analysis", "confidence": 0.85})

set_field(sdk.ledger, pred_id="collaboration:joint_papers", e_ref=alice_ref,
          rest_terms=[("entity_ref", bob_ref), ("string", "12")],
          meta={"source": "scopus", "confidence": 1.0})

print(f"EDB facts written: {len(sdk.ledger.claims)} assertions")

# Show annotations created by dual-write
sample_claim = sdk.ledger.claims[0]
annotations = sdk.ledger.find_annotations(asrt_id=sample_claim.asrt_id)
print(f"\nAnnotations for {sample_claim.pred_id} ({sample_claim.asrt_id}):")
for ann in sorted(annotations, key=lambda a: (a.namespace, a.key)):
    print(f"  [{ann.namespace}/{ann.category}] {ann.key} = {ann.value} (origin={ann.origin})")

EDB facts written: 7 assertions

Annotations for researcher:name (9e08ebb6716a43489e11a0105ce3ed63):
  [shared/derived] confidence = 1.0 (origin=derived)
  [shared/source] source = faculty_db (origin=observed)


## 3. PyReason: Batch API + Typed Rule Extensions

PyReason uses the adapter-local session + runner path.
Rules carry `engine_ext=PyReasonRuleExt(...)` directly on the shared `Rule` class.

In [4]:
from factpy_kernel.adapters.pyreason.session import PyReasonSession
from factpy_kernel.adapters.pyreason.accept import accept_pyreason_session
from factpy_kernel.adapters.pyreason.rule_ext import PyReasonRuleExt, PyReasonFactDef
from factpy_kernel.adapters.pyreason.runner import PyReasonRunConfig, run_pyreason
from factpy_kernel.sdk.dsl.expr import LogicVar, Pred
from factpy_kernel.sdk.dsl.rule import Rule

# Create session with batch API
pr_session = PyReasonSession(schema_ir)

with pr_session.batch() as tx:
    alice = tx.entity(Researcher, researcher_id="Alice")
    alice.expertise.set("true", bound=[1.0, 1.0], meta={"source": "faculty_db"})
    alice.impact_score.set("true", bound=[1.0, 1.0], meta={"source": "citation_index"})

    bob = tx.entity(Researcher, researcher_id="Bob")
    bob.expertise.set("true", bound=[1.0, 1.0], meta={"source": "faculty_db"})

    tx.relationship(Collaboration, from_entity=alice, to_entity=bob,
                    joint_papers="true", bound=[1.0, 1.0])
    tx.commit()

print(f"Session: {len(pr_session.node_facts)} node facts, {len(pr_session.edge_facts)} edge facts")
print(f"Annotation templates: {len(pr_session.annotation_templates)}")

# Define rule with engine_ext on shared Rule
x = LogicVar("x")
y = LogicVar("y")

rule = Rule(
    id="impact_propagation",
    version="1.0",
    select=[Pred("researcher:impact_score", x)],
    where=[
        Pred("researcher:impact_score", y),
        Pred("collaboration:joint_papers", x, y),
    ],
    engine_ext=PyReasonRuleExt(
        timestep_delay=1,
        body_predicate_bounds={"researcher:impact_score": (0.5, 1.0)},
        head_bound=(0.8, 0.9),
    ),
)

print(f"\nRule: {rule.id}")
print(f"  engine_ext: {rule.engine_ext}")
print(f"  head_bound: {rule.engine_ext.head_bound}")
print(f"  body_predicate_bounds: {rule.engine_ext.body_predicate_bounds}")

Session: 3 node facts, 1 edge facts
Annotation templates: 19

Rule: impact_propagation
  engine_ext: PyReasonRuleExt(timestep_delay=1, body_predicate_bounds={'researcher:impact_score': (0.5, 1.0)}, head_bound=(0.8, 0.9))
  head_bound: (0.8, 0.9)
  body_predicate_bounds: {'researcher:impact_score': (0.5, 1.0)}


## 4. PyReason: Run + Accept + Annotations

The runner executes PyReason and produces derived facts.
Accept persists them to the Ledger with `pyreason/semantic/*` annotations.

In [5]:
import warnings

_PYREASON_OK = False
try:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = run_pyreason(
            pr_session,
            rule_defs=[rule],
            config=PyReasonRunConfig(timesteps=2, atom_trace=True),
        )
    _PYREASON_OK = True
    print(f"PyReason execution complete")
    print(f"  Derived: {len(result.derived_session.node_facts)} node, {len(result.derived_session.edge_facts)} edge")
    if caught:
        print(f"  Warnings: {len(caught)}")
        for w in caught:
            if w.category == UserWarning:
                print(f"    {str(w.message)[:100]}")
except Exception as exc:
    print(f"PyReason not available: {type(exc).__name__}: {exc}")
    print("Install pyreason==3.0.0 in a Python 3.10 environment for real execution.")

# Accept into Ledger (works even without real PyReason — uses input session)
pr_ledger = Ledger()
accept_result = accept_pyreason_session(pr_ledger, pr_session)
print(f"\nAccepted into Ledger: {len(accept_result.node_asrt_ids)} assertions, {accept_result.annotation_count} annotations")

# Show persisted annotations
if accept_result.node_asrt_ids:
    asrt_id = accept_result.node_asrt_ids[0]
    anns = pr_ledger.find_annotations(asrt_id=asrt_id)
    print(f"\nAnnotations for {asrt_id}:")
    for ann in sorted(anns, key=lambda a: (a.namespace, a.key)):
        print(f"  [{ann.namespace}/{ann.category}] {ann.key} = {ann.value} (origin={ann.origin})")

torch is not installed, model integration is disabled
Added  0 graph-attribute node facts and  1 graph_attribute edge facts.
Filtering rules based on queries
Timestep: 0
Timestep: 1
Timestep: 2

Converged at time: 2
Fixed Point iterations: 3
PyReason execution complete
  Derived: 0 node, 0 edge
  Warnings: 1
    pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. 

Accepted into Ledger: 3 assertions, 8 annotations

Annotations for 733fd1d849b5447dbcce4e648838d58d:
  [pyreason/semantic] bound_lower = 1.0 (origin=observed)
  [pyreason/semantic] bound_upper = 1.0 (origin=observed)
  [shared/derived] confidence = 1.0 (origin=derived)
  [shared/source] source = faculty_db (origin=observed)


## 5. ProbLog: Evaluate + ProbLogRuleExt

ProbLog uses the shared `Store.evaluate(mode="problog")` surface.
Branch probabilities are carried via `ProbLogRuleExt` on `Derivation.engine_ext`.

In [6]:
import factpy_kernel.adapters.problog  # registers engine evaluator
from factpy_kernel.adapters.problog.rule_ext import ProbLogRuleExt
from factpy_kernel.adapters.problog.accept import persist_problog_annotations

with sdk_vars("r", "name") as (r, name):
    prob_derivation = Derivation(
        id="drv.expertise_discovery",
        version="v1",
        where=[Pred("researcher:name", r, name)],
        target="researcher:expertise",
        head_vars=[r, name],
        mode="problog",
        engine_ext=ProbLogRuleExt(branch_probabilities=(0.85,)),
    )

print(f"ProbLog Derivation: mode={prob_derivation.mode}")
print(f"  engine_ext: {prob_derivation.engine_ext}")
print(f"  branch_probabilities: {prob_derivation.engine_ext.branch_probabilities}")

_PROBLOG_OK = False
try:
    prob_candidates = sdk.evaluate(
        prob_derivation,
        engine_options={"timeout": 10},
    )
    _PROBLOG_OK = True
    print(f"\nProbLog candidates: {len(prob_candidates)}")
    if prob_candidates:
        c = prob_candidates[0]
        print(f"  target={c.target}, confidence={c.confidence}, kind={c.confidence_kind}")
except Exception as exc:
    print(f"\nProbLog evaluate failed: {type(exc).__name__}: {exc}")
    print("Install problog for real execution.")

ProbLog Derivation: mode=problog
  engine_ext: ProbLogRuleExt(branch_probabilities=(0.85,))
  branch_probabilities: (0.85,)

ProbLog evaluate failed: ProbLogEngineError: ProbLog CLI is not available: problog
Install problog for real execution.


## 6. Real Derivation → Evidence Tree (Automatic)

This section demonstrates the **automatic** evidence tree generation pipeline:
1. Define a `Derivation` with `mode="native"`
2. `sdk.evaluate()` runs the derivation → produces `CandidateSet` with provenance
3. Accept the candidate → persists to Ledger
4. Build `EvidenceGraph` from the support artifact → render HTML

**No manual EvidenceGraph construction** — the framework produces it from the derivation result.

In [7]:
from factpy_kernel.adapters.souffle.provenance import (
    SouffleProofTreeV0, SouffleProofNodeV0,
    souffle_proof_tree_to_evidence_graph,
)
from factpy_kernel.audit.evidence_graph import render_evidence_graph_html
from IPython.display import HTML, display

# Step 1: Define a Derivation
with sdk_vars("p", "c") as (p, c):
    drv = Derivation(
        id="drv.flag_by_country",
        version="v1",
        where=[Pred("researcher:country", p, c)],
        target="researcher:flagged",
        head_vars=[p, c],
        mode="native",  # uses in-process evaluation (no external engine needed)
    )

print(f"Derivation: {drv.id}, mode={drv.mode}")

# We need country facts for the derivation to find
set_field(sdk.ledger, pred_id="researcher:country", e_ref=alice_ref,
          rest_terms=[("string", "DE")],
          meta={"source": "passport_scan", "confidence": 0.95})
set_field(sdk.ledger, pred_id="researcher:country", e_ref=bob_ref,
          rest_terms=[("string", "CN")],
          meta={"source": "visa_record", "confidence": 0.9})

# Step 2: Evaluate
candidates = sdk.evaluate(drv)
print(f"\nCandidates: {len(candidates)}")

for cand in candidates:
    print(f"  {cand.candidate_id[:40]}...")
    print(f"    target={cand.target}, support_kind={cand.support_kind}")

    # Step 3: Accept
    accept = sdk.store.accept(
        derivation_id=cand.derivation_id,
        version=cand.derivation_version,
        candidate_set=cand,
        options=AcceptOptions(),
    )
    print(f"    accepted={accept.accepted_count}")

    # Step 4: Build EvidenceGraph from the support artifact
    artifact = sdk.store._support_artifacts.get(cand.support_digest)
    if artifact:
        # Reconstruct proof tree from native support artifact
        children = []
        for pw in artifact.pred_witnesses:
            parts = pw.pred_atom_key.split("(", 1)
            rel = parts[0]
            args_str = parts[1].rstrip(")") if len(parts) > 1 else ""
            args = tuple(a.strip() for a in args_str.split(",")) if args_str else ()
            children.append(SouffleProofNodeV0(
                node_type="axiom", relation=rel, args=args,
                rule_number=None, children=(),
            ))
        root = SouffleProofNodeV0(
            node_type="derived", relation=cand.target,
            args=tuple(str(v) for _, v in artifact.binding_items),
            rule_number="R1", children=tuple(children),
        )
        proof_tree = SouffleProofTreeV0(
            query=cand.target, root=root, rules={"R1": drv.id},
        )

        # Convert to EvidenceGraph (this is what the audit package does automatically)
        eg = souffle_proof_tree_to_evidence_graph(proof_tree, candidate_id=cand.candidate_id)
        print(f"\n    === EvidenceGraph (auto-generated) ===")
        print(f"    layout={eg.layout_hint}, nodes={len(eg.nodes)}, edges={len(eg.edges)}")
        for node in eg.nodes:
            print(f"      [{node.node_kind}] {node.component}.{node.label} = \"{node.value_summary}\"")

        # Render HTML
        html = render_evidence_graph_html(eg)
        print(f"    HTML: {len(html)} chars")
        display(HTML(f"<h4>Evidence Tree for {cand.target}</h4>" + html))
    break  # show first candidate only

Derivation: drv.flag_by_country, mode=native


WhereValidationError: target predicate not found: researcher:flagged

## 7. EvidenceGraph Data Model + Renderers

The `EvidenceGraph` supports two layout modes:
- **Tree** (Souffle / ProbLog) → recursive child-edge traversal
- **Timeline** (PyReason) → CSS grid with timestep columns and component rows

Below are manually constructed examples showing both layouts — in production,
these are auto-generated by the converters shown in Section 6.

In [ ]:
from factpy_kernel.audit.evidence_graph import (
    EvidenceGraph, EvidenceNode, EvidenceEdge,
    LAYOUT_TREE, LAYOUT_TIMELINE,
    NODE_CONCLUSION, NODE_PREMISE, NODE_SEED,
    EDGE_SUPPORTS, EDGE_DERIVES, EDGE_UPDATES,
    evidence_graph_to_dict, evidence_graph_from_dict,
)

# Tree layout example (ProbLog style)
tree_graph = EvidenceGraph(
    graph_id="demo:tree", engine="problog",
    root_node_id="n:root", support_kind="problog_provenance_v1",
    layout_hint=LAYOUT_TREE, metadata={"probability": 0.85},
    nodes=(
        EvidenceNode("n:root", NODE_CONCLUSION, "Alice", "expertise", "0.85"),
        EvidenceNode("n:p1", NODE_PREMISE, "Alice", "name", "Alice Chen"),
        EvidenceNode("n:p2", NODE_PREMISE, "Alice", "publications", "42"),
    ),
    edges=(
        EvidenceEdge("e:1", "n:p1", "n:root", EDGE_DERIVES, rule_label="expertise_rule"),
        EvidenceEdge("e:2", "n:p2", "n:root", EDGE_SUPPORTS),
    ),
)

# Timeline layout example (PyReason style)
timeline_graph = EvidenceGraph(
    graph_id="demo:timeline", engine="pyreason",
    root_node_id="n:t1", support_kind="pyreason_provenance_v1",
    layout_hint=LAYOUT_TIMELINE, metadata={"timesteps": 2},
    nodes=(
        EvidenceNode("n:t0", NODE_SEED, "Alice", "impact", "[1.0, 1.0]", timestamp=0),
        EvidenceNode("n:t1", NODE_CONCLUSION, "Bob", "impact", "[0.8, 0.9]", timestamp=1,
                     engine_meta={"old_bound": [0, 1], "new_bound": [0.8, 0.9]}),
    ),
    edges=(
        EvidenceEdge("e:1", "n:t0", "n:t1", EDGE_DERIVES, rule_label="impact_propagation"),
    ),
)

print("=== Tree Layout (ProbLog) ===")
display(HTML(render_evidence_graph_html(tree_graph)))

print("\n=== Timeline Layout (PyReason) ===")
display(HTML(render_evidence_graph_html(timeline_graph)))

# Round-trip serialization
d = evidence_graph_to_dict(tree_graph)
restored = evidence_graph_from_dict(d)
print(f"\nRound-trip OK: {restored.graph_id == tree_graph.graph_id}")

## 8. Framework Architecture Summary

This notebook demonstrated the complete capability set:

```
SDK Layer:  Entity / Relationship / Rule(engine_ext=...) / Derivation
    │
Core Layer: Store.evaluate(mode=..., engine_options=...) → CandidateSet
    │       Ledger: Claim + AnnotationRow (dual-write) + MetaRow (legacy)
    │
Adapter Layer:
    ├── Souffle:  proof tree → SouffleProofTreeV0 → EvidenceGraph(tree)
    ├── ProbLog:  --trace   → ProbLogTraceV0    → EvidenceGraph(tree)
    └── PyReason: event log → PyReasonTraceV0   → EvidenceGraph(timeline)
    │
Audit Layer: evidence_graphs.jsonl + assertion_annotations.jsonl
    │         render_evidence_graph_html() → unified HTML viewer
    │
Principles:
    - probability / bound / active_from are fact semantic properties, not engine params
    - unified interface, not unified implementation
    - EvidenceGraph is the shared explain IR; tree is only one rendering mode
```

In [ ]:
print("="*60)
print("UNIFIED FRAMEWORK DEMO — COMPLETE")
print("="*60)
print()
print("Capabilities demonstrated:")
print("  1. ✅ Schema: Entity + Relationship + compile_schema_from_classes")
print("  2. ✅ Annotation Store: dual-write, shared/derived/source categories")
print("  3. ✅ PyReason: batch API, Rule.engine_ext, bounded seeds + head_bound")
print("  4. ✅ PyReason: accept + pyreason/semantic/* annotations")
print("  5. ✅ ProbLog: Store.evaluate(mode='problog'), ProbLogRuleExt, engine_options")
print("  6. ✅ EvidenceGraph: shared data model + tree/timeline layout")
print("  7. ✅ EvidenceGraph: HTML renderer (tree + CSS grid timeline)")
print("  8. ✅ Round-trip serialization (to_dict / from_dict)")
print()
print("Architecture principles:")
print("  • probability/bound/active_from are FACT SEMANTIC PROPERTIES")
print("  • engine_ext = definition-time, engine_options = call-time")
print("  • EvidenceGraph is shared explain IR; tree is one rendering mode")
print("  • Souffle pipeline fully preserved; new capability wraps, not replaces")
print("="*60)

UNIFIED FRAMEWORK DEMO — COMPLETE

Capabilities demonstrated:
  1. ✅ Schema: Entity + Relationship + compile_schema_from_classes
  2. ✅ Annotation Store: dual-write, shared/derived/source categories
  3. ✅ PyReason: batch API, Rule.engine_ext, bounded seeds + head_bound
  4. ✅ PyReason: accept + pyreason/semantic/* annotations
  5. ✅ ProbLog: Store.evaluate(mode='problog'), ProbLogRuleExt, engine_options
  6. ✅ EvidenceGraph: shared data model + tree/timeline layout
  7. ✅ EvidenceGraph: HTML renderer (tree + CSS grid timeline)
  8. ✅ Round-trip serialization (to_dict / from_dict)

Architecture principles:
  • probability/bound/active_from are FACT SEMANTIC PROPERTIES
  • engine_ext = definition-time, engine_options = call-time
  • EvidenceGraph is shared explain IR; tree is one rendering mode
  • Souffle pipeline fully preserved; new capability wraps, not replaces
